[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-10-operating-agents-retry-storms-and-budgets.ipynb)

# Advanced Discussion 10 — retry storms, load shedding, Little's law, error budgets and cost tails

A queueing simulation of retries during a backend stall, plus concurrency, availability-budget and heavy-tailed cost calculations.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in about a minute.

In [ ]:
%pip install -q numpy

In [ ]:
import math, numpy as np
rng = np.random.RandomState(0)

## 1. retry storms: a shared model endpoint with fixed capacity, clients that retry on timeout

In [ ]:
CAP = 100                                                           # requests the backend can finish per second
def simulate(arrival_rate, policy, seconds=240, timeout=2.0, stall=(60, 80), seed=0, shed=False):
    """FIFO backend finishing CAP requests per second, except during a stall (an upstream provider outage) when it finishes none.
    A client whose request has waited longer than `timeout` gives up and, if the policy allows, sends a retry - but the abandoned
    request still sits in the queue and consumes capacity when it reaches the front. Returns goodput, load amplification and the
    number of seconds after the stall ends until the queue is back under one second of work."""
    r = np.random.RandomState(seed); queue = []; retry_due = []; done_origin = set(); n_origin = 0; offered = 0; oid = 0; recovered_at = None; retries_this_sec = 0
    def backoff(k): return 0 if policy == "immediate" else (2 ** k) * (0.5 + r.rand())
    for t in range(seconds):
        arrivals = []
        for _ in range(r.poisson(arrival_rate)): arrivals.append(dict(o=oid, k=1, t0=t, retried=False)); oid += 1; n_origin += 1
        due_now = [x for x in retry_due if x[0] <= t]; retry_due = [x for x in retry_due if x[0] > t]
        for due, req in due_now: arrivals.append(dict(o=req["o"], k=req["k"] + 1, t0=t, retried=False))
        offered += len(arrivals); queue += arrivals
        cap_now = 0 if stall[0] <= t < stall[1] else CAP
        if shed:                                                               # deadline-aware backend: drop work whose client already gave up
            queue = [q for q in queue if t - q['t0'] + 1.0 <= timeout]
        served, queue = queue[:cap_now], queue[cap_now:]
        for req in served:
            if t - req["t0"] + 1.0 <= timeout: done_origin.add(req["o"])
        budget = int(0.1 * len(arrivals)) + 1                                   # 'budget' policy: retries limited to ~10% of current traffic
        for req in queue:
            if not req["retried"] and t - req["t0"] + 1.0 > timeout and req["k"] < 3 and policy != "none":
                if policy == "budget":
                    if budget <= 0: continue
                    budget -= 1
                req["retried"] = True; retry_due.append((t + int(math.ceil(backoff(req["k"]) if policy != "budget" else (2 ** req["k"]) * (0.5 + r.rand()))), req))
        queue = [q for q in queue if t - q["t0"] < 30]
        if t >= stall[1] and recovered_at is None and len(queue) < CAP: recovered_at = t - stall[1]
    return len(done_origin) / n_origin, offered / n_origin, recovered_at
print("backend capacity 100 req/s, steady arrivals 80 req/s, a 20-second stall of the backend (seconds 60-80); clients time out after 2 s, retry up to 3 times")
print("policy                          goodput   load amplification   seconds to drain the backlog after the stall")
for pol, label, shed in (("none", "no retries", False), ("immediate", "immediate retries", False), ("backoff", "exponential backoff + jitter", False), ("budget", "backoff + retry budget (10%)", False),
                         ("none", "no retries + drop expired work", True), ("budget", "retry budget + drop expired work", True)):
    g, amp, rec = simulate(80, pol, shed=shed); print("%-30s  %.3f     %.2fx                %s" % (label, g, amp, "never (within the run)" if rec is None else f"{rec} s"))

## 2. Little's law and concurrency: how many agent runs are in flight?

In [ ]:
lam, W = 4.0, 45.0                                                   # 4 new agent tasks per second, each taking 45 s end to end
print("\nLittle's law: L = lambda x W = %.0f x %.0f = %.0f agent runs in flight at once; at 6 model calls per run and 2.0 s per call that is about %.0f concurrent model calls" % (lam, W, lam * W, lam * W * 6 * 2.0 / W))

## 3. error budgets: what an availability target actually allows

In [ ]:
for slo in (0.99, 0.995, 0.999):
    print("SLO %.1f%% over 30 days -> %.1f hours of allowed failure (%.0f minutes)" % (100 * slo, 30 * 24 * (1 - slo), 30 * 24 * 60 * (1 - slo)))
# for an agent, 'failure' includes wrong answers: budget in tasks
n_tasks = 100000
for sr in (0.90, 0.95, 0.99): print("task success target %.0f%% of %d tasks/month -> %d failed tasks allowed" % (100 * sr, n_tasks, round(n_tasks * (1 - sr))))

## 4. cost per task is heavy-tailed: budget by percentile, not by average

In [ ]:
tokens = rng.lognormal(mean=math.log(6000), sigma=0.8, size=100000)         # tokens per task
price = 2.0 / 1e6                                                            # PLACEHOLDER dollars per token
cost = tokens * price
print("\ncost per task ($, placeholder price): mean %.4f | p50 %.4f | p95 %.4f | p99 %.4f | max %.4f" % (cost.mean(), *np.percentile(cost, [50, 95, 99]), cost.max()))
print("the most expensive 5%% of tasks account for %.0f%% of total spend" % (100 * np.sort(cost)[-5000:].sum() / cost.sum()))
cap = np.percentile(cost, 99); print("a hard per-task cap at the p99 (%.4f) would have stopped %d of %d tasks and saved %.1f%% of spend" % (cap, (cost > cap).sum(), len(cost), 100 * (cost[cost > cap] - cap).sum() / cost.sum()))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.